In [1]:
from pathlib import Path

PROJECT_ROOT = Path(
    r"D:\Documents\Ml project\Bangladesh_License_Plate_Recognition"
)

SOURCE_DATASET = (
    PROJECT_ROOT
    / "dataset"
    / "yolo"
    / "License Plate Localization"
)

IMPROVED_DATASET = (
    PROJECT_ROOT
    / "dataset"
    / "yolo"
    / "License Plate Localization Improved"
)

print("LOCALIZATION IMPROVEMENT WORKSPACE")
print("==================================")
print("Project:", PROJECT_ROOT)
print("Source dataset:", SOURCE_DATASET)
print("Source exists:", SOURCE_DATASET.exists())
print("Improved dataset:", IMPROVED_DATASET)
print("Improved dataset exists:", IMPROVED_DATASET.exists())

LOCALIZATION IMPROVEMENT WORKSPACE
Project: D:\Documents\Ml project\Bangladesh_License_Plate_Recognition
Source dataset: D:\Documents\Ml project\Bangladesh_License_Plate_Recognition\dataset\yolo\License Plate Localization
Source exists: True
Improved dataset: D:\Documents\Ml project\Bangladesh_License_Plate_Recognition\dataset\yolo\License Plate Localization Improved
Improved dataset exists: False


In [2]:
def folder_size_bytes(folder):
    total = 0

    for file in folder.rglob("*"):
        if file.is_file():
            total += file.stat().st_size

    return total


source_size = folder_size_bytes(SOURCE_DATASET)

print("SOURCE DATASET SIZE")
print("===================")
print("Bytes:", source_size)
print("GB:", round(source_size / (1024 ** 3), 2))

SOURCE DATASET SIZE
Bytes: 1695653727
GB: 1.58


In [3]:
IMPROVEMENT_ROOT = (
    PROJECT_ROOT
    / "dataset"
    / "localization_improvement"
)

NEW_IMAGES = IMPROVEMENT_ROOT / "images"
NEW_LABELS = IMPROVEMENT_ROOT / "labels"
REPORTS = IMPROVEMENT_ROOT / "reports"

for folder in [
    IMPROVEMENT_ROOT,
    NEW_IMAGES,
    NEW_LABELS,
    REPORTS
]:
    folder.mkdir(parents=True, exist_ok=True)

print("IMPROVEMENT WORKSPACE CREATED")
print("=============================")
print("Root:", IMPROVEMENT_ROOT)
print("Images:", NEW_IMAGES)
print("Labels:", NEW_LABELS)
print("Reports:", REPORTS)

print("\nOriginal dataset unchanged:")
print(SOURCE_DATASET.exists())

IMPROVEMENT WORKSPACE CREATED
Root: D:\Documents\Ml project\Bangladesh_License_Plate_Recognition\dataset\localization_improvement
Images: D:\Documents\Ml project\Bangladesh_License_Plate_Recognition\dataset\localization_improvement\images
Labels: D:\Documents\Ml project\Bangladesh_License_Plate_Recognition\dataset\localization_improvement\labels
Reports: D:\Documents\Ml project\Bangladesh_License_Plate_Recognition\dataset\localization_improvement\reports

Original dataset unchanged:
True


In [4]:
from PIL import Image

TRAIN_IMAGES = SOURCE_DATASET / "images" / "train"
TRAIN_LABELS = SOURCE_DATASET / "labels" / "train"

image_extensions = {
    ".jpg", ".jpeg", ".png", ".jfif", ".bmp", ".webp"
}

image_map = {
    p.stem: p
    for p in TRAIN_IMAGES.iterdir()
    if p.suffix.lower() in image_extensions
}

small_plate_images = []

for label_path in TRAIN_LABELS.glob("*.txt"):

    text = label_path.read_text(
        encoding="utf-8"
    ).strip()

    if not text or label_path.stem not in image_map:
        continue

    image_path = image_map[label_path.stem]

    with Image.open(image_path) as img:
        img_w, img_h = img.size

    smallest_width = float("inf")
    smallest_height = float("inf")

    for line in text.splitlines():

        parts = line.split()

        if len(parts) != 5:
            continue

        _, xc, yc, w, h = map(float, parts)

        plate_w = w * img_w
        plate_h = h * img_h

        smallest_width = min(
            smallest_width,
            plate_w
        )

        smallest_height = min(
            smallest_height,
            plate_h
        )

    if (
        smallest_width < 100
        or smallest_height < 50
    ):
        small_plate_images.append(
            (
                image_path,
                smallest_width,
                smallest_height
            )
        )

print("SMALL-PLATE TRAINING IMAGES")
print("===========================")
print("Found:", len(small_plate_images))

print("\nFirst 10 examples:")

for image_path, w, h in small_plate_images[:10]:
    print(
        image_path.name,
        "| plate:",
        f"{w:.1f} x {h:.1f} px"
    )

SMALL-PLATE TRAINING IMAGES
Found: 627

First 10 examples:
10_jpg.rf.bb971b38897a1a8b3e5dad0d66bf556c.jpg | plate: 97.0 x 68.0 px
10_jpg.rf.c3bbb5e1926fc3e17de4a324f33e7fd5.jpg | plate: 97.0 x 68.0 px
10_jpg.rf.f0d3c76c0392942db023dbdec6a59279.jpg | plate: 97.0 x 68.0 px
121_jpg.rf.bbfe0ed798cb7b5c127b122bb87c2f90.jpg | plate: 97.0 x 53.0 px
121_jpg.rf.bcc76f06354e530ccd612c3226947e43.jpg | plate: 97.0 x 53.0 px
121_jpg.rf.cfac65902907db65b0d0438350d43bf6.jpg | plate: 97.0 x 53.0 px
123_jpg.rf.5bc5da9809e58935a7e68dbef90dc103.jpg | plate: 80.0 x 51.0 px
123_jpg.rf.950382dd76dc8d1ec4619c14f96c3629.jpg | plate: 80.0 x 51.0 px
123_jpg.rf.9d3c07d9331236b896bf816fae2588aa.jpg | plate: 80.0 x 51.0 px
123_jpg.rf.a1e6b08b6345913269db72ed976a9c80.jpg | plate: 80.0 x 51.0 px


In [5]:
from collections import Counter

def source_prefix(path):
    name = path.stem

    if ".rf." in name:
        return name.split(".rf.")[0]

    return name

all_source_counts = Counter(
    source_prefix(path)
    for path in image_map.values()
)

small_source_counts = Counter(
    source_prefix(record[0])
    for record in small_plate_images
)

print("SOURCE-PREFIX ANALYSIS")
print("======================")

print("Training images:", len(image_map))
print("Unique source prefixes:", len(all_source_counts))

print("\nSources with multiple training images:")
print(sum(count > 1 for count in all_source_counts.values()))

print("\nMaximum variants from one source:")
print(max(all_source_counts.values()))

print("\nSmall-plate images:", len(small_plate_images))
print(
    "Unique small-plate source prefixes:",
    len(small_source_counts)
)

print("\nMost repeated small-plate sources:")

for source, count in small_source_counts.most_common(15):
    print(source, "->", count, "images")

SOURCE-PREFIX ANALYSIS
Training images: 5124
Unique source prefixes: 2376

Sources with multiple training images:
940

Maximum variants from one source:
8

Small-plate images: 627
Unique small-plate source prefixes: 268

Most repeated small-plate sources:
123_jpg -> 4 images
126_jpg -> 4 images
128_jpg -> 4 images
12_jpg -> 4 images
134_jpg -> 4 images
138_jpg -> 4 images
152333555_261628735361740_2714349024038729117_n--1-_jpg -> 4 images
154292220_233596325124901_5741689577336570017_n--1-_jpg -> 4 images
16_jpg -> 4 images
17_jpg -> 4 images
205_png -> 4 images
21_jpg -> 4 images
24_jpg -> 4 images
26_jpg -> 4 images
29_jpg -> 4 images


In [6]:
variant_distribution = Counter(all_source_counts.values())

print("TRAINING SOURCE VARIANT DISTRIBUTION")
print("====================================")

for variant_count in sorted(variant_distribution):
    source_count = variant_distribution[variant_count]
    image_count = variant_count * source_count

    print(
        f"{variant_count} image(s) per source:"
        f" {source_count} sources"
        f" -> {image_count} images"
    )

print("\nSummary:")
print("Total training images:", sum(all_source_counts.values()))
print("Unique source prefixes:", len(all_source_counts))

single_sources = sum(
    count == 1
    for count in all_source_counts.values()
)

repeated_sources = sum(
    count > 1
    for count in all_source_counts.values()
)

print("Sources appearing once:", single_sources)
print("Sources appearing multiple times:", repeated_sources)

TRAINING SOURCE VARIANT DISTRIBUTION
1 image(s) per source: 1436 sources -> 1436 images
2 image(s) per source: 14 sources -> 28 images
3 image(s) per source: 99 sources -> 297 images
4 image(s) per source: 804 sources -> 3216 images
5 image(s) per source: 11 sources -> 55 images
7 image(s) per source: 4 sources -> 28 images
8 image(s) per source: 8 sources -> 64 images

Summary:
Total training images: 5124
Unique source prefixes: 2376
Sources appearing once: 1436
Sources appearing multiple times: 940


In [7]:
rf_images = [
    path
    for path in image_map.values()
    if ".rf." in path.stem
]

non_rf_images = [
    path
    for path in image_map.values()
    if ".rf." not in path.stem
]

print("ROBOFLOW-STYLE VARIANT ANALYSIS")
print("===============================")

print("Total training images:", len(image_map))
print("Images containing '.rf.':", len(rf_images))
print("Images without '.rf.':", len(non_rf_images))

print("\nPercentages:")

print(
    ".rf. images:",
    round(len(rf_images) / len(image_map) * 100, 2),
    "%"
)

print(
    "Non-.rf. images:",
    round(len(non_rf_images) / len(image_map) * 100, 2),
    "%"
)

ROBOFLOW-STYLE VARIANT ANALYSIS
Total training images: 5124
Images containing '.rf.': 5124
Images without '.rf.': 0

Percentages:
.rf. images: 100.0 %
Non-.rf. images: 0.0 %


In [8]:
SAFE_DATASET = (
    PROJECT_ROOT
    / "dataset"
    / "final_leakage_safe"
    / "License Plate Localization"
)

print("LEAKAGE-SAFE SOURCE DATASET")
print("===========================")
print("Path:", SAFE_DATASET)
print("Exists:", SAFE_DATASET.exists())

if SAFE_DATASET.exists():
    for split in ["train", "validation", "test"]:
        split_path = SAFE_DATASET / split

        files = [
            p for p in split_path.rglob("*")
            if p.is_file()
        ] if split_path.exists() else []

        images = [
            p for p in files
            if p.suffix.lower()
            in {".jpg", ".jpeg", ".png", ".jfif"}
        ]

        xmls = [
            p for p in files
            if p.suffix.lower() == ".xml"
        ]

        print(
            f"{split}:",
            len(images), "images,",
            len(xmls), "XML"
        )

LEAKAGE-SAFE SOURCE DATASET
Path: D:\Documents\Ml project\Bangladesh_License_Plate_Recognition\dataset\final_leakage_safe\License Plate Localization
Exists: True
train: 5124 images, 5124 XML
validation: 640 images, 640 XML
test: 641 images, 641 XML


In [9]:
IMPROVEMENT_CONFIG = {
    "train_images": 5124,
    "keep_validation_unchanged": True,
    "keep_test_unchanged": True,

    "target_conditions": [
        "low_light",
        "brightness_variation",
        "blur",
        "low_resolution",
        "scale_distance",
        "perspective"
    ],

    "avoid": [
        "heavy_augmentation",
        "validation_augmentation",
        "test_augmentation",
        "blind_duplicate_generation"
    ]
}

config_file = REPORTS / "improvement_plan.txt"

text = """LOCALIZATION IMPROVEMENT PLAN

Keep:
- Original leakage-safe training set
- Original validation set unchanged
- Original test set unchanged

Targeted training improvements:
1. Low-light variation
2. Brightness/contrast variation
3. Mild blur
4. Low-resolution simulation
5. Scale/distance variation
6. Mild perspective variation

Important:
Do not augment validation or test data.
Do not blindly generate variants from every training image.
Preserve YOLO bounding boxes correctly.
Prioritize difficult and underrepresented conditions.

Primary goal:
Improve real-world localization generalization while preserving
the leakage-safe evaluation protocol.
"""

config_file.write_text(text, encoding="utf-8")

print("IMPROVEMENT PLAN CREATED")
print("========================")
print(config_file)
print("Exists:", config_file.exists())

IMPROVEMENT PLAN CREATED
D:\Documents\Ml project\Bangladesh_License_Plate_Recognition\dataset\localization_improvement\reports\improvement_plan.txt
Exists: True


In [10]:
import random

random.seed(42)

unique_small_sources = {}

for image_path, plate_w, plate_h in small_plate_images:
    prefix = source_prefix(image_path)

    if prefix not in unique_small_sources:
        unique_small_sources[prefix] = (
            image_path,
            plate_w,
            plate_h
        )

selected_difficult = list(unique_small_sources.values())

print("TARGETED IMPROVEMENT SELECTION")
print("==============================")
print("Small-plate images before source filtering:", len(small_plate_images))
print("Unique difficult sources selected:", len(selected_difficult))

print("\nFirst 10 selected sources:")

for image_path, w, h in selected_difficult[:10]:
    print(
        image_path.name,
        "| plate:",
        f"{w:.1f} x {h:.1f} px"
    )

TARGETED IMPROVEMENT SELECTION
Small-plate images before source filtering: 627
Unique difficult sources selected: 268

First 10 selected sources:
10_jpg.rf.bb971b38897a1a8b3e5dad0d66bf556c.jpg | plate: 97.0 x 68.0 px
121_jpg.rf.bbfe0ed798cb7b5c127b122bb87c2f90.jpg | plate: 97.0 x 53.0 px
123_jpg.rf.5bc5da9809e58935a7e68dbef90dc103.jpg | plate: 80.0 x 51.0 px
126_jpg.rf.735d25bbfadf8d509fccb4505ec32476.jpg | plate: 85.0 x 50.0 px
128_jpg.rf.0f35b6e47567c756cb550cd12b4c59c2.jpg | plate: 87.0 x 49.0 px
12_jpg.rf.2191b8041c43dbe3195bc52c50f2476f.jpg | plate: 91.0 x 68.0 px
130_jpg.rf.3d69eed96117e5290b7ef9ff32516852.jpg | plate: 98.0 x 47.0 px
133_jpg.rf.22fcc1b0a4dab45a8caf08321a48f01d.jpg | plate: 75.0 x 41.0 px
134_jpg.rf.6150187f7062f65a8c7c4e940a5a427b.jpg | plate: 101.0 x 50.0 px
135_jpg.rf.2d0ee7c302300b2e5733e21bcc02138e.jpg | plate: 107.0 x 50.0 px


In [11]:
from PIL import ImageEnhance
import shutil

AUG_IMAGES = NEW_IMAGES / "low_light"
AUG_LABELS = NEW_LABELS / "low_light"

AUG_IMAGES.mkdir(parents=True, exist_ok=True)
AUG_LABELS.mkdir(parents=True, exist_ok=True)

created = 0

for image_path, plate_w, plate_h in selected_difficult:
    label_path = TRAIN_LABELS / f"{image_path.stem}.txt"

    if not label_path.exists():
        continue

    with Image.open(image_path) as img:
        image = img.convert("RGB")

        dark_image = ImageEnhance.Brightness(
            image
        ).enhance(0.55)

        output_image = (
            AUG_IMAGES
            / f"{image_path.stem}_lowlight.jpg"
        )

        dark_image.save(
            output_image,
            quality=95
        )

    output_label = (
        AUG_LABELS
        / f"{image_path.stem}_lowlight.txt"
    )

    shutil.copy2(
        label_path,
        output_label
    )

    created += 1

print("LOW-LIGHT AUGMENTATION COMPLETE")
print("===============================")
print("Created images:", created)
print("Created labels:", len(list(AUG_LABELS.glob("*.txt"))))
print("Image folder:", AUG_IMAGES)
print("Label folder:", AUG_LABELS)

LOW-LIGHT AUGMENTATION COMPLETE
Created images: 268
Created labels: 268
Image folder: D:\Documents\Ml project\Bangladesh_License_Plate_Recognition\dataset\localization_improvement\images\low_light
Label folder: D:\Documents\Ml project\Bangladesh_License_Plate_Recognition\dataset\localization_improvement\labels\low_light


In [12]:
LOWRES_IMAGES = NEW_IMAGES / "low_resolution"
LOWRES_LABELS = NEW_LABELS / "low_resolution"

LOWRES_IMAGES.mkdir(parents=True, exist_ok=True)
LOWRES_LABELS.mkdir(parents=True, exist_ok=True)

created = 0

for image_path, plate_w, plate_h in selected_difficult:
    label_path = TRAIN_LABELS / f"{image_path.stem}.txt"

    if not label_path.exists():
        continue

    with Image.open(image_path) as img:
        image = img.convert("RGB")

        original_size = image.size

        small_size = (
            max(1, original_size[0] // 3),
            max(1, original_size[1] // 3)
        )

        degraded = image.resize(
            small_size,
            Image.Resampling.BILINEAR
        )

        degraded = degraded.resize(
            original_size,
            Image.Resampling.BILINEAR
        )

        output_image = (
            LOWRES_IMAGES
            / f"{image_path.stem}_lowres.jpg"
        )

        degraded.save(
            output_image,
            quality=85
        )

    output_label = (
        LOWRES_LABELS
        / f"{image_path.stem}_lowres.txt"
    )

    shutil.copy2(
        label_path,
        output_label
    )

    created += 1

print("LOW-RESOLUTION AUGMENTATION COMPLETE")
print("====================================")
print("Created images:", created)
print("Created labels:", len(list(LOWRES_LABELS.glob("*.txt"))))

LOW-RESOLUTION AUGMENTATION COMPLETE
Created images: 268
Created labels: 268


In [13]:
augmentation_sets = [
    ("low_light", AUG_IMAGES, AUG_LABELS),
    ("low_resolution", LOWRES_IMAGES, LOWRES_LABELS)
]

total_images = 0
total_labels = 0
missing_labels = []

for name, image_dir, label_dir in augmentation_sets:

    images = list(image_dir.glob("*.jpg"))
    labels = list(label_dir.glob("*.txt"))

    print(name.upper())
    print("Images:", len(images))
    print("Labels:", len(labels))

    suffix = "_lowlight" if name == "low_light" else "_lowres"

    for image_path in images:
        label_path = label_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            missing_labels.append(image_path.name)

    total_images += len(images)
    total_labels += len(labels)

print("\nTOTAL")
print("Images:", total_images)
print("Labels:", total_labels)
print("Missing labels:", len(missing_labels))

if not missing_labels:
    print("\nSTATUS: AUGMENTATION DATA VERIFIED")
else:
    print("\nSTATUS: CHECK REQUIRED")

LOW_LIGHT
Images: 268
Labels: 268
LOW_RESOLUTION
Images: 268
Labels: 268

TOTAL
Images: 536
Labels: 536
Missing labels: 0

STATUS: AUGMENTATION DATA VERIFIED


In [14]:
import shutil

total, used, free = shutil.disk_usage(PROJECT_ROOT)

print("DISK SPACE")
print("==========")
print("Total:", round(total / (1024**3), 2), "GB")
print("Used:", round(used / (1024**3), 2), "GB")
print("Free:", round(free / (1024**3), 2), "GB")

DISK SPACE
Total: 137.89 GB
Used: 91.09 GB
Free: 46.79 GB


In [15]:
import shutil

if IMPROVED_DATASET.exists():
    shutil.rmtree(IMPROVED_DATASET)

print("Copying original dataset...")

shutil.copytree(
    SOURCE_DATASET,
    IMPROVED_DATASET
)

improved_train_images = IMPROVED_DATASET / "images" / "train"
improved_train_labels = IMPROVED_DATASET / "labels" / "train"

print("Adding targeted augmentations...")

for _, image_dir, label_dir in augmentation_sets:

    for image_path in image_dir.glob("*.jpg"):
        shutil.copy2(
            image_path,
            improved_train_images / image_path.name
        )

    for label_path in label_dir.glob("*.txt"):
        shutil.copy2(
            label_path,
            improved_train_labels / label_path.name
        )

print("\nIMPROVED DATASET CREATED")
print("========================")

for split in ["train", "validation", "test"]:

    image_dir = IMPROVED_DATASET / "images" / split
    label_dir = IMPROVED_DATASET / "labels" / split

    print(
        split,
        "-> images:",
        len(list(image_dir.iterdir())),
        "| labels:",
        len(list(label_dir.glob("*.txt")))
    )

Copying original dataset...
Adding targeted augmentations...

IMPROVED DATASET CREATED
train -> images: 5660 | labels: 5660
validation -> images: 640 | labels: 640
test -> images: 641 | labels: 641


In [16]:
invalid_labels = []
missing_labels = []
total_boxes = 0

for split in ["train", "validation", "test"]:

    image_dir = IMPROVED_DATASET / "images" / split
    label_dir = IMPROVED_DATASET / "labels" / split

    images = [
        p for p in image_dir.iterdir()
        if p.suffix.lower() in image_extensions
    ]

    for image_path in images:

        label_path = label_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            missing_labels.append(str(image_path))
            continue

        text = label_path.read_text(
            encoding="utf-8"
        ).strip()

        if not text:
            continue

        for line in text.splitlines():

            parts = line.split()

            if len(parts) != 5:
                invalid_labels.append(str(label_path))
                continue

            cls, xc, yc, w, h = map(float, parts)

            if not (
                0 <= xc <= 1
                and 0 <= yc <= 1
                and 0 < w <= 1
                and 0 < h <= 1
            ):
                invalid_labels.append(str(label_path))

            total_boxes += 1

print("IMPROVED DATASET VERIFICATION")
print("=============================")
print("Total boxes:", total_boxes)
print("Missing labels:", len(missing_labels))
print("Invalid labels:", len(set(invalid_labels)))

if not missing_labels and not invalid_labels:
    print("\nSTATUS: READY FOR TRAINING")
else:
    print("\nSTATUS: FIX REQUIRED")

IMPROVED DATASET VERIFICATION
Total boxes: 5289
Missing labels: 0
Invalid labels: 0

STATUS: READY FOR TRAINING


In [17]:
ZIP_BASE = PROJECT_ROOT / "localization_yolo_improved"

print("Creating ZIP...")

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=IMPROVED_DATASET
)

zip_size = Path(zip_path).stat().st_size / (1024 ** 3)

print("\nTRAINING PACKAGE CREATED")
print("========================")
print("ZIP:", zip_path)
print("Size:", round(zip_size, 2), "GB")
print("Exists:", Path(zip_path).exists())

Creating ZIP...

TRAINING PACKAGE CREATED
ZIP: D:\Documents\Ml project\Bangladesh_License_Plate_Recognition\localization_yolo_improved.zip
Size: 1.65 GB
Exists: True
